# 스티어링 벡터 벤치마크

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/modulabs-personalab/psyctl/blob/main/examples/ko/06_benchmark_vectors.ipynb)

이 노트북에서는 IPIP-NEO-120 성격 검사를 사용하여 사전 학습된 BiPO 스티어링 벡터를 체계적으로 평가합니다. 각 벡터가 다양한 강도에서 Big Five 성격 프로파일을 어떻게 변화시키는지 측정합니다.

**이 노트북에서 다루는 내용:**
1. 사전 학습된 5개 영어 BiPO 벡터를 다운로드합니다
2. 여러 강도에서 IPIP-NEO-120 벤치마크를 실행합니다
3. 교차 영향(각 벡터가 모든 Big Five 영역에 미치는 영향)을 분석합니다
4. (선택사항) OpenRouter로 API 모델을 테스트합니다

**요구사항:** Colab 무료 티어(T4 GPU). Llama-3.1-8B-Instruct 접근 권한이 있는 [HuggingFace 토큰](https://huggingface.co/settings/tokens).

**소요 시간:** 약 15분

In [ ]:
!pip install -q git+https://github.com/modulabs-personalab/psyctl.git bitsandbytes accelerate

In [ ]:
import os

try:
    from google.colab import userdata
    _hf = userdata.get("HF_TOKEN")
    os.environ["HF_TOKEN"] = _hf if isinstance(_hf, str) else str(_hf)
    print("Loaded HF_TOKEN from Colab Secrets")
except Exception:
    if not os.environ.get("HF_TOKEN"):
        raise RuntimeError(
            "HF_TOKEN not found. Please:\n"
            "1. Click the key icon (Secrets) in the left sidebar\n"
            "2. Add HF_TOKEN with your HuggingFace token\n"
            "3. Toggle 'Notebook access' ON for HF_TOKEN\n"
            "4. Re-run this cell"
        )

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PSYCTL_LOG_LEVEL"] = "WARNING"

# Optional: OpenRouter API key (for the API model section at the end)
try:
    from google.colab import userdata
    _or = userdata.get("OPENROUTER_API_KEY")
    os.environ["OPENROUTER_API_KEY"] = _or if isinstance(_or, str) else str(_or)
    print("Loaded OPENROUTER_API_KEY from Colab Secrets")
except Exception:
    pass

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PSYCTL_LOG_LEVEL"] = "WARNING"

## 사전 학습된 벡터 다운로드

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path

VECTOR_REPO = "dalekwon/bipo-steering-vectors"
VECTOR_DIR = Path("./vectors")
VECTOR_DIR.mkdir(exist_ok=True)

# Big Five-mapped vectors for inventory benchmarking
VECTORS = {
    "agreeableness": {
        "filename": "bipo_steering_english_agreeableness.safetensors",
        "big_five_domain": "A",
    },
    "neuroticism": {
        "filename": "bipo_steering_english_neuroticism.safetensors",
        "big_five_domain": "N",
    },
}

# Additional personality vectors
EXTRA_VECTORS = {
    "awfully_sweet": "bipo_steering_english_awfully_sweet.safetensors",
    "paranoid": "bipo_steering_english_paranoid.safetensors",
    "very_lascivious": "bipo_steering_english_very_lascivious.safetensors",
}

vector_paths = {}
for name, info in VECTORS.items():
    vector_paths[name] = Path(hf_hub_download(
        repo_id=VECTOR_REPO, filename=info["filename"],
        local_dir=str(VECTOR_DIR), token=os.environ.get("HF_TOKEN"),
    ))
    print(f"  {name} (Big Five: {info['big_five_domain']}): ready")

for name, filename in EXTRA_VECTORS.items():
    vector_paths[name] = Path(hf_hub_download(
        repo_id=VECTOR_REPO, filename=filename,
        local_dir=str(VECTOR_DIR), token=os.environ.get("HF_TOKEN"),
    ))
    print(f"  {name}: ready")

print(f"\nAll {len(vector_paths)} vectors downloaded.")

## IPIP-NEO-120 벤치마크

PSYCTL의 `InventoryTester`를 사용하여 여러 스티어링 강도에서 전체 IPIP-NEO-120 검사를 실행합니다. 스티어링 벡터를 적용했을 때 각 Big Five 영역이 어떻게 변화하는지 측정합니다.

In [ ]:
from psyctl.core.benchmark.inventory_tester import InventoryTester

MODEL = "meta-llama/Llama-3.1-8B-Instruct"
STRENGTHS = [1.0, 2.0, 3.0]
INVENTORY = "ipip_neo_120"

tester = InventoryTester()
all_results = {}

# Benchmark Big Five-mapped vectors
for trait_name, info in VECTORS.items():
    vpath = vector_paths[trait_name]
    print(f"\n{'='*60}")
    print(f"Benchmarking: {trait_name} (target domain: {info['big_five_domain']})")
    print(f"{'='*60}")

    trait_results = []
    for strength in STRENGTHS:
        print(f"  Testing strength={strength}...")
        result = tester.test_inventory(
            model=MODEL,
            steering_vector_path=vpath,
            inventory_name=INVENTORY,
            steering_strength=strength,
            target_trait=trait_name,
        )
        trait_results.append({"strength": strength, "result": result})

        # Print comparison
        comparison = result.get("comparison", {})
        if comparison:
            for code, d in comparison.items():
                marker = " <--" if code == info["big_five_domain"] else ""
                print(f"    {d['domain_name']:>20}: {d['change']:+.3f} ({d['percent_change']:+.1f}%){marker}")

    all_results[trait_name] = trait_results

print("\nBenchmark complete!")

## 교차 영향 분석

좋은 스티어링 벡터는 목표 Big Five 영역에 주로 영향을 미치면서 다른 영역에 대한 의도하지 않은 변화를 최소화해야 합니다. 이를 시각화해봅니다.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Extract cross-impact data at strength=2.0
domains = ["N", "E", "O", "A", "C"]
domain_names = {"N": "Neuroticism", "E": "Extraversion", "O": "Openness",
                "A": "Agreeableness", "C": "Conscientiousness"}

fig, axes = plt.subplots(1, len(VECTORS), figsize=(6 * len(VECTORS), 5), sharey=True)
if len(VECTORS) == 1:
    axes = [axes]

for idx, (trait_name, info) in enumerate(VECTORS.items()):
    ax = axes[idx]

    # Find strength=2.0 result
    for r in all_results[trait_name]:
        if r["strength"] == 2.0:
            comparison = r["result"].get("comparison", {})
            break
    else:
        comparison = {}

    changes = []
    colors = []
    for d in domains:
        if d in comparison:
            changes.append(comparison[d]["percent_change"])
            colors.append("#E85D75" if d == info["big_five_domain"] else "#4A90D9")
        else:
            changes.append(0)
            colors.append("#cccccc")

    ax.bar(range(len(domains)), changes, color=colors)
    ax.set_xticks(range(len(domains)))
    ax.set_xticklabels([domain_names[d] for d in domains], rotation=30, ha="right")
    ax.set_title(f"{trait_name} (target: {info['big_five_domain']})", fontweight="bold")
    ax.set_ylabel("% Change from Baseline" if idx == 0 else "")
    ax.axhline(y=0, color="grey", linewidth=0.5)

plt.suptitle("Cross-Impact: Strength = 2.0", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## (선택사항) OpenRouter를 활용한 검사 테스트

OpenRouter를 사용하여 여러 API 모델을 테스트합니다. 성격 검사 질문을 채팅 API로 전송하고 리커트 척도 응답을 파싱합니다.

In [ ]:
# Uncomment to run OpenRouter inventory test on API models
#
# import re, time
# from psyctl.data.inventories import create_inventory
# from psyctl.models.openrouter_client import OpenRouterClient
#
# OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
# if not OPENROUTER_API_KEY:
#     print("Set OPENROUTER_API_KEY in Colab Secrets to use this section.")
# else:
#     client = OpenRouterClient(api_key=OPENROUTER_API_KEY)
#     inventory = create_inventory("rei_40")
#     questions = inventory.get_questions()
#
#     SYSTEM_PROMPT = (
#         "You are taking a personality assessment. For each statement, respond with ONLY "
#         "a single number from 1 to 5.\n\nScale:\n1 = Definitely not true\n"
#         "2 = Somewhat not true\n3 = Neither true nor untrue\n4 = Somewhat true\n"
#         "5 = Definitely true\n\nRespond with ONLY the number."
#     )
#
#     MODELS = [
#         ("anthropic/claude-sonnet-4", "Claude Sonnet 4"),
#         ("openai/gpt-4o", "GPT-4o"),
#     ]
#
#     for model_id, model_name in MODELS:
#         print(f"\nTesting {model_name}...")
#         domain_responses = {}
#         for q in questions:
#             prompt = f'Statement: "{q["text"]}"\nYour rating (1-5):'
#             _, resp = client.generate(
#                 prompt=prompt, model=model_id,
#                 temperature=0, max_tokens=16, system_prompt=SYSTEM_PROMPT,
#             )
#             match = re.search(r"\b([1-5])\b", resp.strip())
#             score = float(match.group(1)) if match else 3.0
#             if q["keyed"] == "minus":
#                 score = 6.0 - score
#             domain = q["domain"]
#             domain_responses.setdefault(domain, []).append(score)
#             time.sleep(0.3)
#
#         scores = inventory.calculate_scores(domain_responses)
#         print(f"  {model_name} REI-40 scores:")
#         for code in sorted(scores.keys()):
#             s = scores[code]
#             print(f"    {s['domain_name']:<25} raw={s['raw_score']:.1f}  z={s['z_score']:+.2f}  pctl={s['percentile']:.1f}%")

## 주요 발견

- **목표 영향력**: BiPO 벡터는 주로 목표 Big Five 영역을 변화시키며, 의도한 성격 방향을 정확히 포착하고 있음을 확인할 수 있습니다
- **교차 영역 효과**: 약간의 교차 영향은 예상 가능합니다. 예를 들어 우호성 벡터가 외향성을 미세하게 증가시킬 수 있습니다
- **강도 스케일링**: 높은 강도는 더 큰 변화를 만들지만, 매우 높은 값(3.0 초과)에서는 응답의 일관성이 떨어질 수 있습니다
- **권장 범위**: 강도 1.0에서 2.5 사이가 성격 변화와 출력 품질 간의 균형이 가장 좋습니다

**다음 단계:**
- [01_quickstart.ipynb](./01_quickstart.ipynb) -- 인터랙티브하게 조향을 체험합니다
- [04_extract_vector.ipynb](./04_extract_vector.ipynb) -- 나만의 벡터를 추출합니다
- [GitHub](https://github.com/modulabs-personalab/psyctl) -- 새로운 검사 도구나 벡터를 기여해주세요